Needed imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# Classifiers
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Oversampling
from imblearn.over_sampling import RandomOverSampler

# Gain Ratio (Information Gain / Entropy) via mutual_info_classif
from sklearn.feature_selection import mutual_info_classif

Loading Dataset

In [2]:

print("=" * 70)
print("SECTION 1: LOADING DATASET")
print("=" * 70)

DATASET_PATH = "feature_vectors_syscallsbinders_frequency_5_Cat.csv"  # <-- Change this to your file path

df = pd.read_csv(DATASET_PATH)
# df = df.drop_duplicates() -> Baseline study didn't explicitly state that they removed the duplicates

print(f"Dataset shape : {df.shape}")
print(f"Columns : {df.columns.tolist()[-5:]} ... (last 5 shown)")
print(f"\nClass distribution:\n{df['Class'].value_counts()}")

# Map class names to match paper labels
label_map = {
    1: "Adware",
    2: "Banking",
    3: "SMS_Malware",
    4: "Riskware",
    5: "Benign"
}

if df["Class"].dtype != object:
    df["Class"] = df["Class"].map(label_map)

print(f"\nMapped class distribution:\n{df['Class'].value_counts()}")

SECTION 1: LOADING DATASET
Dataset shape : (11598, 471)
Columns : ['watchRotation', 'windowGainedFocus', 'write', 'writev', 'Class'] ... (last 5 shown)

Class distribution:
Class
3    3904
4    2546
2    2100
5    1795
1    1253
Name: count, dtype: int64

Mapped class distribution:
Class
SMS_Malware    3904
Riskware       2546
Banking        2100
Benign         1795
Adware         1253
Name: count, dtype: int64


Basic Preprocessing — Feature/Label Split

In [3]:

print("\n" + "=" * 70)
print("SECTION 2: BASIC PREPROCESSING — FEATURE/LABEL SPLIT")
print("=" * 70)

X = df.drop(columns=["Class"])
y = df["Class"]

# Encode string labels to integers for sklearn
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Feature matrix shape : {X.shape}")
print(f"Label classes        : {le.classes_}")


SECTION 2: BASIC PREPROCESSING — FEATURE/LABEL SPLIT
Feature matrix shape : (11598, 470)
Label classes        : ['Adware' 'Banking' 'Benign' 'Riskware' 'SMS_Malware']


Basic Preprocessing — Standard Scaling

In [4]:

print("\n" + "=" * 70)
print("SECTION 3: BASIC PREPROCESSING — STANDARD SCALING")
print("=" * 70)

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print(f"Scaling done. Mean ~ 0, Std ~ 1 verified:")
print(f"  Column means range : [{X_scaled.mean().min():.4f}, {X_scaled.mean().max():.4f}]")
print(f"  Column stds range  : [{X_scaled.std().min():.4f}, {X_scaled.std().max():.4f}]")


SECTION 3: BASIC PREPROCESSING — STANDARD SCALING
Scaling done. Mean ~ 0, Std ~ 1 verified:
  Column means range : [-0.0000, 0.0000]
  Column stds range  : [1.0000, 1.0000]


Preprocessing — Random Oversampling (ROA)

In [5]:

print("\n" + "=" * 70)
print("SECTION 4: PREPROCESSING — RANDOM OVERSAMPLING (ROA)")
print("=" * 70)

print(f"Before ROA — class distribution:\n{pd.Series(y_encoded).value_counts().sort_index()}")

ros = RandomOverSampler(random_state=42)
X_scaled_ros, y_ros = ros.fit_resample(X_scaled, y_encoded)
X_scaled_ros = pd.DataFrame(X_scaled_ros, columns=X_scaled.columns)

print(f"\nAfter ROA — class distribution:\n{pd.Series(y_ros).value_counts().sort_index()}")
print(f"\nDataset size before ROA : {X_scaled.shape}")
print(f"Dataset size after ROA  : {X_scaled_ros.shape}")


SECTION 4: PREPROCESSING — RANDOM OVERSAMPLING (ROA)
Before ROA — class distribution:
0    1253
1    2100
2    1795
3    2546
4    3904
Name: count, dtype: int64

After ROA — class distribution:
0    3904
1    3904
2    3904
3    3904
4    3904
Name: count, dtype: int64

Dataset size before ROA : (11598, 470)
Dataset size after ROA  : (19520, 470)


Feature Selection — Gain Ratio

In [6]:

print("\n" + "=" * 70)
print("SECTION 5: FEATURE SELECTION — GAIN RATIO")
print("=" * 70)


def compute_entropy(series, bins=20):
    """Compute the entropy H(X) of a continuous feature via binning."""
    counts, _ = np.histogram(series, bins=bins)
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))


def gain_ratio_scores(X_df, y_arr):
    """
    Compute gain ratio for each feature.
    Gain Ratio(X; Y) = IG(X; Y) / H(X)
    Returns a Series of gain ratio scores indexed by feature name.
    """
    ig_scores = mutual_info_classif(X_df, y_arr, random_state=42)
    h_scores = np.array([compute_entropy(X_df.iloc[:, i])
                         for i in range(X_df.shape[1])])
    gr_scores = np.where(h_scores > 0, ig_scores / h_scores, 0.0)
    return pd.Series(gr_scores, index=X_df.columns)


print("Computing gain ratio scores on ROA-balanced data (this may take a minute)...")
gr = gain_ratio_scores(X_scaled_ros, y_ros)

selected_features = gr[gr > 0].index.tolist()
zero_features     = gr[gr == 0].index.tolist()

print(f"\nTotal features         : {len(gr)}")
print(f"Features with GR > 0   : {len(selected_features)}")
print(f"Features removed (GR=0): {len(zero_features)}")
print(f"Reduction %            : {len(zero_features)/len(gr)*100:.1f}%")

top10 = gr.nlargest(10)
print(f"\nTop 10 features by Gain Ratio:\n{top10.to_string()}")

# Plot Top 10 Gain Ratio Features
fig, ax = plt.subplots(figsize=(8, 5))
top10.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Gain Ratio Score")
ax.set_title("Top 10 Features by Gain Ratio")
plt.tight_layout()
plt.savefig("figure2_gain_ratio_top10.png", dpi=150)
plt.close()
print("Saved: figure2_gain_ratio_top10.png")

# Apply feature selection to ROA-balanced data
X_selected_ros = X_scaled_ros[selected_features]


SECTION 5: FEATURE SELECTION — GAIN RATIO
Computing gain ratio scores on ROA-balanced data (this may take a minute)...

Total features         : 470
Features with GR > 0   : 390
Features removed (GR=0): 80
Reduction %            : 17.0%

Top 10 features by Gain Ratio:
wait4                 108.052911
writev                 55.233238
NETWORK_ACCESS____     51.495390
windowGainedFocus      48.032476
sched_yield            44.808885
poll                   33.542718
setpriority            31.554265
getNetworkInfo         25.617977
DEVICE_ACCESS_____     25.546489
dup                    22.758605
Saved: figure2_gain_ratio_top10.png


Train/Test Split

In [7]:

print("\n" + "=" * 70)
print("SECTION 6: TRAIN/TEST SPLIT (80/20)")
print("=" * 70)

# --- Without gain ratio (all 470 ROA features) ---
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_scaled_ros, y_ros, test_size=0.2, random_state=42, stratify=y_ros
)

# --- With gain ratio (selected features only) ---
X_train_sel, X_test_sel, _, _ = train_test_split(
    X_selected_ros, y_ros, test_size=0.2, random_state=42, stratify=y_ros
)

print(f"Train size (full)     : {X_train_full.shape}")
print(f"Test size  (full)     : {X_test_full.shape}")
print(f"Train size (selected) : {X_train_sel.shape}")
print(f"Test size  (selected) : {X_test_sel.shape}")


SECTION 6: TRAIN/TEST SPLIT (80/20)
Train size (full)     : (15616, 470)
Test size  (full)     : (3904, 470)
Train size (selected) : (15616, 390)
Test size  (selected) : (3904, 390)


Single Model Classification

In [8]:

print("\n" + "=" * 70)
print("SECTION 7: SINGLE MODEL CLASSIFICATION")
print("=" * 70)

models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Extra Tree"   : ExtraTreesClassifier(random_state=42),
    "Naive Bayes"  : GaussianNB(),
    "KNN"          : KNeighborsClassifier(),
    "SVM"          : SVC(random_state=42),
}


def evaluate(y_true, y_pred):
    """Return accuracy, precision, recall, f1 as percentages."""
    return {
        "Accuracy" : accuracy_score(y_true, y_pred) * 100,
        "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0) * 100,
        "Recall"   : recall_score(y_true, y_pred, average="weighted", zero_division=0) * 100,
        "F1-Score" : f1_score(y_true, y_pred, average="weighted", zero_division=0) * 100,
    }


results_single = {}

for name, model in models.items():
    print(f"\n  Training {name}...")

    # Without gain ratio
    model.fit(X_train_full, y_train)
    pred_full    = model.predict(X_test_full)
    metrics_full = evaluate(y_test, pred_full)

    # With gain ratio
    model.fit(X_train_sel, y_train)
    pred_sel    = model.predict(X_test_sel)
    metrics_sel = evaluate(y_test, pred_sel)

    results_single[name] = {"without_gr": metrics_full, "with_gr": metrics_sel}

    for metric in ["Accuracy", "Precision", "Recall", "F1-Score"]:
        diff = metrics_sel[metric] - metrics_full[metric]
        print(f"    {metric:10s} | Without GR: {metrics_full[metric]:.2f}%"
              f" | With GR: {metrics_sel[metric]:.2f}%"
              f" | Δ: {diff:+.2f}%")

# Replicate Figures 6-9
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-Score"]
fig_nums        = [6, 7, 8, 9]

for metric, fig_num in zip(metrics_to_plot, fig_nums):
    model_names = list(results_single.keys())
    before = [results_single[m]["without_gr"][metric] for m in model_names]
    after  = [results_single[m]["with_gr"][metric]    for m in model_names]

    x     = np.arange(len(model_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 4))
    bars1 = ax.barh(x - width/2, before, width, label="Before feature selection", color="#1f77b4")
    bars2 = ax.barh(x + width/2, after,  width, label="After feature selection",  color="#ff7f0e")

    ax.set_yticks(x)
    ax.set_yticklabels(model_names)
    ax.set_xlabel(f"{metric} (%)")
    ax.set_title(f"Figure {fig_num}. Model {metric} Before and After Gain Ratio")
    ax.legend()
    ax.set_xlim(40, 100)

    for bar in bars1:
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f"{bar.get_width():.2f}%", va="center", fontsize=8)
    for bar in bars2:
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f"{bar.get_width():.2f}%", va="center", fontsize=8)

    plt.tight_layout()
    fname = f"figure{fig_num}_model_{metric.lower().replace('-','')}.png"
    plt.savefig(fname, dpi=150)
    plt.close()
    print(f"Saved: {fname}")


SECTION 7: SINGLE MODEL CLASSIFICATION

  Training Random Forest...
    Accuracy   | Without GR: 98.57% | With GR: 98.59% | Δ: +0.03%
    Precision  | Without GR: 98.59% | With GR: 98.61% | Δ: +0.03%
    Recall     | Without GR: 98.57% | With GR: 98.59% | Δ: +0.03%
    F1-Score   | Without GR: 98.56% | With GR: 98.59% | Δ: +0.03%

  Training Extra Tree...
    Accuracy   | Without GR: 98.57% | With GR: 98.54% | Δ: -0.03%
    Precision  | Without GR: 98.60% | With GR: 98.56% | Δ: -0.03%
    Recall     | Without GR: 98.57% | With GR: 98.54% | Δ: -0.03%
    F1-Score   | Without GR: 98.56% | With GR: 98.54% | Δ: -0.03%

  Training Naive Bayes...
    Accuracy   | Without GR: 56.38% | With GR: 57.38% | Δ: +1.00%
    Precision  | Without GR: 65.30% | With GR: 66.86% | Δ: +1.56%
    Recall     | Without GR: 56.38% | With GR: 57.38% | Δ: +1.00%
    F1-Score   | Without GR: 55.32% | With GR: 56.41% | Δ: +1.09%

  Training KNN...


Exception in thread Thread-3 (_readerthread):
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3568.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3568.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3568.0_x64__qbz5n2kfra8p0\Lib\subprocess.py", line 1615, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3568.0_x64__qbz5n2kfra8p0\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Unic

    Accuracy   | Without GR: 91.09% | With GR: 91.21% | Δ: +0.13%
    Precision  | Without GR: 91.27% | With GR: 91.39% | Δ: +0.12%
    Recall     | Without GR: 91.09% | With GR: 91.21% | Δ: +0.13%
    F1-Score   | Without GR: 91.06% | With GR: 91.19% | Δ: +0.13%

  Training SVM...
    Accuracy   | Without GR: 82.22% | With GR: 82.22% | Δ: +0.00%
    Precision  | Without GR: 83.33% | With GR: 83.26% | Δ: -0.07%
    Recall     | Without GR: 82.22% | With GR: 82.22% | Δ: +0.00%
    F1-Score   | Without GR: 82.33% | With GR: 82.33% | Δ: -0.00%
Saved: figure6_model_accuracy.png
Saved: figure7_model_precision.png
Saved: figure8_model_recall.png
Saved: figure9_model_f1score.png


Ensemble Voting (Hard voting)

In [9]:

print("\n" + "=" * 70)
print("SECTION 8: ENSEMBLE VOTING CLASSIFICATION")
print("=" * 70)

rf  = RandomForestClassifier(random_state=42)
et  = ExtraTreesClassifier(random_state=42)
nb  = GaussianNB()
knn = KNeighborsClassifier()
svm = SVC(random_state=42)

ensemble_combos = {
    "RF, ET, k-NN"         : [("RF", rf), ("ET", et), ("KNN", knn)],
    "RF, ET, SVM"          : [("RF", rf), ("ET", et), ("SVM", svm)],
    "RF, ET, NB"           : [("RF", rf), ("ET", et), ("NB",  nb)],
    "RF, k-NN, SVM"        : [("RF", rf), ("KNN", knn), ("SVM", svm)],
    "RF, k-NN, NB"         : [("RF", rf), ("KNN", knn), ("NB",  nb)],
    "RF, SVM, NB"          : [("RF", rf), ("SVM", svm), ("NB",  nb)],
    "ET, k-NN, SVM"        : [("ET", et), ("KNN", knn), ("SVM", svm)],
    "ET, k-NN, NB"         : [("ET", et), ("KNN", knn), ("NB",  nb)],
    "k-NN, SVM, NB"        : [("KNN", knn), ("SVM", svm), ("NB", nb)],
    "RF, ET, k-NN, SVM, NB": [("RF", rf), ("ET", et), ("KNN", knn), ("SVM", svm), ("NB", nb)],
}

ensemble_results = {}

for combo_name, estimators in ensemble_combos.items():
    print(f"\n  Ensemble: {combo_name}")

    vc_full = VotingClassifier(estimators=estimators, voting="hard")
    vc_full.fit(X_train_full, y_train)
    pred_full    = vc_full.predict(X_test_full)
    metrics_full = evaluate(y_test, pred_full)

    vc_sel = VotingClassifier(estimators=estimators, voting="hard")
    vc_sel.fit(X_train_sel, y_train)
    pred_sel    = vc_sel.predict(X_test_sel)
    metrics_sel = evaluate(y_test, pred_sel)

    ensemble_results[combo_name] = {"without_gr": metrics_full, "with_gr": metrics_sel}

    print(f"    WITHOUT GR → Acc: {metrics_full['Accuracy']:.2f}%"
          f"  Prec: {metrics_full['Precision']:.2f}%"
          f"  Rec: {metrics_full['Recall']:.2f}%"
          f"  F1: {metrics_full['F1-Score']:.2f}%")
    print(f"    WITH GR    → Acc: {metrics_sel['Accuracy']:.2f}%"
          f"  Prec: {metrics_sel['Precision']:.2f}%"
          f"  Rec: {metrics_sel['Recall']:.2f}%"
          f"  F1: {metrics_sel['F1-Score']:.2f}%")


SECTION 8: ENSEMBLE VOTING CLASSIFICATION

  Ensemble: RF, ET, k-NN
    WITHOUT GR → Acc: 98.49%  Prec: 98.53%  Rec: 98.49%  F1: 98.49%
    WITH GR    → Acc: 98.51%  Prec: 98.55%  Rec: 98.51%  F1: 98.51%

  Ensemble: RF, ET, SVM
    WITHOUT GR → Acc: 98.51%  Prec: 98.56%  Rec: 98.51%  F1: 98.51%
    WITH GR    → Acc: 98.54%  Prec: 98.58%  Rec: 98.54%  F1: 98.54%

  Ensemble: RF, ET, NB
    WITHOUT GR → Acc: 98.54%  Prec: 98.59%  Rec: 98.54%  F1: 98.54%
    WITH GR    → Acc: 98.54%  Prec: 98.58%  Rec: 98.54%  F1: 98.54%

  Ensemble: RF, k-NN, SVM
    WITHOUT GR → Acc: 94.72%  Prec: 94.99%  Rec: 94.72%  F1: 94.74%
    WITH GR    → Acc: 94.60%  Prec: 94.86%  Rec: 94.60%  F1: 94.61%

  Ensemble: RF, k-NN, NB
    WITHOUT GR → Acc: 94.16%  Prec: 94.63%  Rec: 94.16%  F1: 94.16%
    WITH GR    → Acc: 94.06%  Prec: 94.49%  Rec: 94.06%  F1: 94.06%

  Ensemble: RF, SVM, NB
    WITHOUT GR → Acc: 87.14%  Prec: 88.49%  Rec: 87.14%  F1: 87.10%
    WITH GR    → Acc: 87.45%  Prec: 88.70%  Rec: 87.45% 

Results Table

In [10]:

print("\n" + "=" * 70)
print("SECTION 9: ENSEMBLE RESULTS TABLE (Table 2 replication)")
print("=" * 70)

rows = []
for combo_name, res in ensemble_results.items():
    rows.append({
        "Models"       : combo_name,
        "Acc (no GR)"  : f"{res['without_gr']['Accuracy']:.2f}",
        "Prec (no GR)" : f"{res['without_gr']['Precision']:.2f}",
        "Rec (no GR)"  : f"{res['without_gr']['Recall']:.2f}",
        "F1 (no GR)"   : f"{res['without_gr']['F1-Score']:.2f}",
        "Acc (with GR)" : f"{res['with_gr']['Accuracy']:.2f}",
        "Prec (with GR)": f"{res['with_gr']['Precision']:.2f}",
        "Rec (with GR)" : f"{res['with_gr']['Recall']:.2f}",
        "F1 (with GR)"  : f"{res['with_gr']['F1-Score']:.2f}",
    })

results_df = pd.DataFrame(rows)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print(results_df.to_string(index=False))

results_df.to_csv("table2_ensemble_results.csv", index=False)
print("\nSaved: table2_ensemble_results.csv")


SECTION 9: ENSEMBLE RESULTS TABLE (Table 2 replication)
               Models Acc (no GR) Prec (no GR) Rec (no GR) F1 (no GR) Acc (with GR) Prec (with GR) Rec (with GR) F1 (with GR)
         RF, ET, k-NN       98.49        98.53       98.49      98.49         98.51          98.55         98.51        98.51
          RF, ET, SVM       98.51        98.56       98.51      98.51         98.54          98.58         98.54        98.54
           RF, ET, NB       98.54        98.59       98.54      98.54         98.54          98.58         98.54        98.54
        RF, k-NN, SVM       94.72        94.99       94.72      94.74         94.60          94.86         94.60        94.61
         RF, k-NN, NB       94.16        94.63       94.16      94.16         94.06          94.49         94.06        94.06
          RF, SVM, NB       87.14        88.49       87.14      87.10         87.45          88.70         87.45        87.41
        ET, k-NN, SVM       94.75        95.01       94.75   

Confusion Matrix

In [11]:

print("\n" + "=" * 70)
print("SECTION 10: CONFUSION MATRIX — BEST ENSEMBLE (RF, ET, k-NN + GR)")
print("=" * 70)

best_ensemble = VotingClassifier(
    estimators=[
        ("RF",  RandomForestClassifier(random_state=42)),
        ("ET",  ExtraTreesClassifier(random_state=42)),
        ("KNN", KNeighborsClassifier()),
    ],
    voting="hard"
)
best_ensemble.fit(X_train_sel, y_train)
best_pred = best_ensemble.predict(X_test_sel)

cm         = confusion_matrix(y_test, best_pred)
class_names = le.classes_

print(f"\nClassification Report:\n")
print(classification_report(y_test, best_pred, target_names=class_names))

per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100
for cls, acc in zip(class_names, per_class_acc):
    print(f"  {cls:15s}: {acc:.2f}%")

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names, ax=ax
)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("Figure 10. Confusion Matrix — RF, ET, k-NN with Gain Ratio")
plt.tight_layout()
plt.savefig("figure10_confusion_matrix.png", dpi=150)
plt.close()
print("\nSaved: figure10_confusion_matrix.png")


SECTION 10: CONFUSION MATRIX — BEST ENSEMBLE (RF, ET, k-NN + GR)

Classification Report:

              precision    recall  f1-score   support

      Adware       0.96      1.00      0.98       781
     Banking       0.99      1.00      0.99       780
      Benign       0.99      0.99      0.99       781
    Riskware       0.99      0.95      0.97       781
 SMS_Malware       1.00      0.99      1.00       781

    accuracy                           0.99      3904
   macro avg       0.99      0.99      0.99      3904
weighted avg       0.99      0.99      0.99      3904

  Adware         : 99.74%
  Banking        : 99.62%
  Benign         : 98.85%
  Riskware       : 95.13%
  SMS_Malware    : 99.23%

Saved: figure10_confusion_matrix.png


Comparison Results

In [12]:

print("\n" + "=" * 70)
print("SECTION 11: COMPARISON WITH BASELINE (Table 3)")
print("=" * 70)

best_metrics = evaluate(y_test, best_pred)

comparison = pd.DataFrame([
    {
        "Model"    : "Nguyen et al. [13] (baseline)",
        "Accuracy" : "97.07",
        "Precision": "95.50",
        "Recall"   : "96.90",
        "F1-Score" : "95.90",
    },
    {
        "Model"    : "Proposed (RF, ET, k-NN + Gain Ratio) [ours]",
        "Accuracy" : f"{best_metrics['Accuracy']:.2f}",
        "Precision": f"{best_metrics['Precision']:.2f}",
        "Recall"   : f"{best_metrics['Recall']:.2f}",
        "F1-Score" : f"{best_metrics['F1-Score']:.2f}",
    },
])
print(comparison.to_string(index=False))
comparison.to_csv("table3_model_comparison.csv", index=False)
print("\nSaved: table3_model_comparison.csv")

print("\n" + "=" * 70)
print("ALL SECTIONS COMPLETE")
print("=" * 70)
print("""
Output files:
  figure2_gain_ratio_top10.png  -> Top 10 features by Gain Ratio
  figure6_model_accuracy.png    -> Accuracy before/after GR
  figure7_model_precision.png   -> Precision before/after GR
  figure8_model_recall.png      -> Recall before/after GR
  figure9_model_f1score.png     -> F1 before/after GR
  figure10_confusion_matrix.png -> Confusion matrix (best model)
  table2_ensemble_results.csv   -> Ensemble voting results (Table 2)
  table3_model_comparison.csv   -> Comparison with baseline (Table 3)
""")


SECTION 11: COMPARISON WITH BASELINE (Table 3)
                                      Model Accuracy Precision Recall F1-Score
              Nguyen et al. [13] (baseline)    97.07     95.50  96.90    95.90
Proposed (RF, ET, k-NN + Gain Ratio) [ours]    98.51     98.55  98.51    98.51

Saved: table3_model_comparison.csv

ALL SECTIONS COMPLETE

Output files:
  figure2_gain_ratio_top10.png  -> Top 10 features by Gain Ratio
  figure6_model_accuracy.png    -> Accuracy before/after GR
  figure7_model_precision.png   -> Precision before/after GR
  figure8_model_recall.png      -> Recall before/after GR
  figure9_model_f1score.png     -> F1 before/after GR
  figure10_confusion_matrix.png -> Confusion matrix (best model)
  table2_ensemble_results.csv   -> Ensemble voting results (Table 2)
  table3_model_comparison.csv   -> Comparison with baseline (Table 3)

